# Survey ETL scenario

The whole pipeline in one process, without a server or Docker: FastAPI's
`TestClient` calls the API directly and the database is a temporary
SQLite file that disappears at the end.

Scenario:

1. Waves 1 and 2 arrive and load cleanly.
2. Wave 3 arrives with problem rows and one late wave 2 respondent.
3. The summaries of waves 2 and 3 are recalculated.

Run it with the project's `.venv` kernel (`make install` sets it up).

In [1]:
import contextlib
import csv
import dataclasses
import io
import re
import tempfile
from pathlib import Path
from typing import Final

import pandas as pd
from fastapi.testclient import TestClient

from survey_etl.api import create_app
from survey_etl.settings import DEFAULT_SETTINGS, Environment

## Set-up

The dev settings with the database moved to a temporary folder. Entering
the client runs the app's lifespan; `cleanup` closes both at the end.

In [2]:
cleanup = contextlib.ExitStack()
database_folder = Path(cleanup.enter_context(tempfile.TemporaryDirectory()))
settings = dataclasses.replace(
    DEFAULT_SETTINGS[Environment.DEV],
    database_url=f"sqlite:///{database_folder / 'survey.db'}",
)
client = cleanup.enter_context(TestClient(create_app(settings)))

client.get("/health").json()

{'status': 'ok', 'environment': 'dev'}

## Helpers

The sample files live in `data/` at the repository root. The helpers
read a CSV file into the request body the way `survey_client` does; the
notebook then posts it as any HTTP client would.

In [3]:
# Whole numbers in ASCII digits only: re's \d alone would also match
# other scripts' digits, which int() accepts.
INTEGER_PATTERN: Final[re.Pattern[str]] = re.compile(r"[+-]?\d+", re.ASCII)

type CellValue = int | str | None
type SurveyRow = dict[str, CellValue]


def parse_cell(
    text: str, integer_pattern: re.Pattern[str] = INTEGER_PATTERN
) -> CellValue:
    """Turn the text of a CSV cell into a JSON value.

    Args:
        text: Cell text as read from the file.
        integer_pattern: Pattern of text that is a whole number.

    Returns:
        ``None`` for an empty or blank cell, an ``int`` for a whole
        number, otherwise the text unchanged, so the service can
        report it.
    """
    stripped_text = text.strip()
    if not stripped_text:
        return None
    if integer_pattern.fullmatch(stripped_text):
        return int(stripped_text)

    return text


def summary_table(api_response: dict) -> pd.DataFrame:
    """Flatten API response to one row per wave, age and brand."""
    brand_rows = [
        {
            "wave": wave["wave"],
            "age category": category["age_category"],
            "respondents": category["respondents_total"],
            "brand": brand["brand"],
            "aware": brand["awareness_total"],
            "users": brand["usage_total"],
        }
        for wave in api_response["waves"]
        for category in wave["age_categories"]
        for brand in category["brands"]
    ]
    return pd.DataFrame(brand_rows)


def respondents_table(summary: pd.DataFrame) -> pd.DataFrame:
    """Respondents per wave (rows) and age category (columns)."""
    return summary.drop_duplicates(["wave", "age category"]).pivot_table(
        index="wave",
        columns="age category",
        values="respondents",
        aggfunc="sum",
        fill_value=0,
        margins=True,
        margins_name="total",
    )


def find_data_folder(start: Path) -> Path:
    """Return the repository's data folder, searching upwards."""
    for folder in (start, *start.parents):
        if (folder / "data").is_dir():
            return folder / "data"
    message = f"no data folder in {start} or its parents"
    raise FileNotFoundError(message)


DATA_PATH: Final[Path] = find_data_folder(Path.cwd())


def read_survey_csv(
    file_name: str, data_path: Path = DATA_PATH
) -> dict[str, list[SurveyRow]]:
    """Read a sample CSV file into the body of POST /batches."""
    # 1. Read file
    csv_text = (data_path / file_name).read_text(encoding="utf-8-sig")
    csv_reader = csv.reader(io.StringIO(csv_text, newline=""))

    # 2. Grab column names
    column_names = next((fields for fields in csv_reader if fields), None)
    if column_names is None:
        raise ValueError(f"{file_name} contains no header row.")
    # TODO: Validate column names here.

    # 3. Grab all rows
    survey_rows: list[SurveyRow] = []
    for fields in csv_reader:
        if not fields:
            continue
        # TODO: Validate data here before appending.
        survey_rows.append(
            dict(zip(column_names, map(parse_cell, fields), strict=True))
        )
    return {"respondents": survey_rows}

## POST new data batch via API

1. Read CSV data file as JSON parsable dictionary.
2. POST the data dictionary.
3. Look into the data base.

In [4]:
survey_data = read_survey_csv("wave1.csv")
survey_data

{'respondents': [{'respondent_id': 1001,
   'interview_date': '02/03/2026',
   'wave': 1,
   'birth_year': 2000,
   'brand1_awareness': 1,
   'brand2_awareness': 1,
   'brand3_awareness': 0,
   'brand1_usage': 1,
   'brand2_usage': 0,
   'brand3_usage': 0},
  {'respondent_id': 1002,
   'interview_date': '02/03/2026',
   'wave': 1,
   'birth_year': 1985,
   'brand1_awareness': 1,
   'brand2_awareness': 0,
   'brand3_awareness': 1,
   'brand1_usage': 1,
   'brand2_usage': 0,
   'brand3_usage': 1},
  {'respondent_id': 1003,
   'interview_date': '03/03/2026',
   'wave': 1,
   'birth_year': 1970,
   'brand1_awareness': 1,
   'brand2_awareness': 1,
   'brand3_awareness': 1,
   'brand1_usage': 0,
   'brand2_usage': 1,
   'brand3_usage': 0},
  {'respondent_id': 1004,
   'interview_date': '03/03/2026',
   'wave': 1,
   'birth_year': 2005,
   'brand1_awareness': 0,
   'brand2_awareness': 1,
   'brand3_awareness': 0,
   'brand1_usage': 0,
   'brand2_usage': 1,
   'brand3_usage': 0},
  {'responden

In [5]:
response = client.post("/batches", json=survey_data)
response.raise_for_status()
response.json()

{'rows_received': 12,
 'rows_loaded': 12,
 'invalid_rows': [],
 'corrected_values': [],
 'refreshed_waves': [1]}

In [6]:
settings.database_url

'sqlite:////tmp/tmps3tm1i3f/survey.db'

Wave 2, the same way:

In [7]:
response = client.post("/batches", json=read_survey_csv("wave2.csv"))
response.raise_for_status()
response.json()

{'rows_received': 12,
 'rows_loaded': 12,
 'invalid_rows': [],
 'corrected_values': [],
 'refreshed_waves': [2]}

## GET aggregated data

1. Run GET on the API to get aggregated data as a dictionary.
2. Convert the data into data frame.

In [8]:
response = client.get("/summary")
response.raise_for_status()
response.json()

{'waves': [{'wave': 1,
   'age_categories': [{'age_category': '18-24',
     'respondents_total': 2,
     'brands': [{'brand': 'brand1', 'awareness_total': 1, 'usage_total': 1},
      {'brand': 'brand2', 'awareness_total': 2, 'usage_total': 1},
      {'brand': 'brand3', 'awareness_total': 1, 'usage_total': 0}]},
    {'age_category': '25-34',
     'respondents_total': 3,
     'brands': [{'brand': 'brand1', 'awareness_total': 3, 'usage_total': 3},
      {'brand': 'brand2', 'awareness_total': 2, 'usage_total': 1},
      {'brand': 'brand3', 'awareness_total': 1, 'usage_total': 0}]},
    {'age_category': '35-44',
     'respondents_total': 2,
     'brands': [{'brand': 'brand1', 'awareness_total': 2, 'usage_total': 1},
      {'brand': 'brand2', 'awareness_total': 0, 'usage_total': 0},
      {'brand': 'brand3', 'awareness_total': 1, 'usage_total': 1}]},
    {'age_category': '45-54',
     'respondents_total': 2,
     'brands': [{'brand': 'brand1', 'awareness_total': 1, 'usage_total': 1},
      {

In [9]:
summary_df = summary_table(response.json())
summary_df

,wave,age category,respondents,brand,aware,users
0,1,18-24,2,brand1,1,1
1,1,18-24,2,brand2,2,1
2,1,18-24,2,brand3,1,0
3,1,25-34,3,brand1,3,3
4,1,25-34,3,brand2,2,1
5,1,25-34,3,brand3,1,0
6,1,35-44,2,brand1,2,1
7,1,35-44,2,brand2,0,0
8,1,35-44,2,brand3,1,1
9,1,45-54,2,brand1,1,1


In [10]:
respondents_table(summary_df)

age category,18-24,25-34,35-44,45-54,55+,total
wave,,,,,,
1,2,3,2,2,3,12
2,2,3,2,2,3,12
total,4,6,4,4,6,24


## Wave 3

The file as received. Rows of interest:

| respondent | problem |
|---|---|
| 3003 | born 2010: 16 in the interview year |
| 1005 | ID already loaded with wave 1 |
| 3006 | `brand1_awareness` is 2 |
| 3007 | uses brand 3 without being aware of it |
| 2013 | late wave 2 interview |

In [11]:
pd.read_csv(DATA_PATH / "wave3.csv", dtype=str)

,respondent_id,interview_date,wave,birth_year,brand1_awareness,brand2_awareness,brand3_awareness,brand1_usage,brand2_usage,brand3_usage
0,3001,07/09/2026,3,2002,1,1,0,1,0,0
1,3002,07/09/2026,3,1986,1,0,1,0,0,1
2,3003,08/09/2026,3,2010,1,1,0,1,1,0
3,3004,08/09/2026,3,1969,1,1,1,1,1,0
4,1005,09/09/2026,3,1978,1,0,0,1,0,0
5,3006,09/09/2026,3,1994,2,1,0,1,0,0
6,3007,10/09/2026,3,1981,1,1,0,1,0,1
7,3008,10/09/2026,3,2005,0,1,1,0,1,0
8,2013,29/06/2026,2,1989,1,1,0,1,0,0
9,3010,11/09/2026,3,1964,1,0,1,1,0,1


Read and POST new data batch via API

In [12]:
survey_data = read_survey_csv("wave3.csv")

response = client.post("/batches", json=survey_data)
response.raise_for_status()
batch_report = response.json()
batch_report

{'rows_received': 12,
 'rows_loaded': 9,
 'invalid_rows': [{'row_number': 3,
   'respondent_id': 3003,
   'values': {'respondent_id': 3003,
    'interview_date': '08/09/2026',
    'wave': 3,
    'birth_year': 2010,
    'brand1_awareness': 1,
    'brand2_awareness': 1,
    'brand3_awareness': 0,
    'brand1_usage': 1,
    'brand2_usage': 1,
    'brand3_usage': 0},
   'reason': 'underage',
   'details': 'age 16 in interview year is below the minimum age 18'},
  {'row_number': 5,
   'respondent_id': 1005,
   'values': {'respondent_id': 1005,
    'interview_date': '09/09/2026',
    'wave': 3,
    'birth_year': 1978,
    'brand1_awareness': 1,
    'brand2_awareness': 0,
    'brand3_awareness': 0,
    'brand1_usage': 1,
    'brand2_usage': 0,
    'brand3_usage': 0},
   'reason': 'duplicated_id',
   'details': 'respondent ID already stored in the database'},
  {'row_number': 6,
   'respondent_id': 3006,
   'values': {'respondent_id': 3006,
    'interview_date': '09/09/2026',
    'wave': 3,
  

In [13]:
settings.database_url

'sqlite:////tmp/tmps3tm1i3f/survey.db'

Rejected rows, with the reason and the values as received:

In [14]:
pd.DataFrame(batch_report["invalid_rows"])[
    ["row_number", "respondent_id", "reason", "details"]
]

,row_number,respondent_id,reason,details
0,3,3003,underage,age 16 in interview year is below the minimum ...
1,5,1005,duplicated_id,respondent ID already stored in the database
2,6,3006,invalid_value,brand1_awareness: Input should be less than or...


Values corrected while loading (usage without awareness is set to 0):

In [15]:
pd.DataFrame(batch_report["corrected_values"])

,row_number,respondent_id,column,received,corrected,reason
0,7,3007,brand3_usage,1,0,usage_without_awareness


Wave 2 now has 13 respondents: the late row was added and its summary
recalculated together with wave 3.

In [16]:
response = client.get("/summary")
response.raise_for_status()

respondents_table(summary_table(response.json()))

age category,18-24,25-34,35-44,45-54,55+,total
wave,,,,,,
1,2,3,2,2,3,12
2,2,3,3,2,3,13
3,2,1,1,2,2,8
total,6,7,6,6,8,33


## 3. Reading summaries

The latest wave only:

In [17]:
response = client.get("/summary", params={"wave_last": 1})
response.raise_for_status()

summary_table(response.json())

,wave,age category,respondents,brand,aware,users
0,3,18-24,2,brand1,1,1
1,3,18-24,2,brand2,2,1
2,3,18-24,2,brand3,1,0
3,3,25-34,1,brand1,1,0
4,3,25-34,1,brand2,1,1
5,3,25-34,1,brand3,1,0
6,3,35-44,1,brand1,1,0
7,3,35-44,1,brand2,0,0
8,3,35-44,1,brand3,1,1
9,3,45-54,2,brand1,1,1


Wave 2, youngest age category (positions count from 1):

In [18]:
response = client.get("/summary", params={"wave": 2, "agecat": 1})
response.raise_for_status()

summary_table(response.json())

,wave,age category,respondents,brand,aware,users
0,2,18-24,2,brand1,1,0
1,2,18-24,2,brand2,2,2
2,2,18-24,2,brand3,2,1


Waves 2 to 3, age categories 2 to 3, brand 1 only:

In [19]:
response = client.get(
    "/summary", params={"wave_from": 2, "agecat_from": 2, "agecat_to": 3}
)
response.raise_for_status()

summary_table(response.json()).query("brand == 'brand1'")

,wave,age category,respondents,brand,aware,users
0,2,25-34,3,brand1,2,2
3,2,35-44,3,brand1,3,2
6,3,25-34,1,brand1,1,0
9,3,35-44,1,brand1,1,0


In [20]:
response.url

URL('http://testserver/summary?wave_from=2&agecat_from=2&agecat_to=3')

## 4. Rejected requests

A wave without data gives 404; two ways of selecting waves at once give
422.

In [21]:
missing_wave = client.get("/summary", params={"wave": 9})
missing_wave.status_code, missing_wave.json()

(404,
 {'detail': 'no stored summary matches the selected waves and age categories'})

In [22]:
two_selections = client.get("/summary", params={"wave": 1, "wave_last": 2})
two_selections.status_code, two_selections.json()

(422,
 {'detail': 'select waves in one way only: a single wave, the latest waves, or a range'})

## Clean-up

Stops the app and deletes the temporary database.

In [23]:
cleanup.close()